# 04 — Tests Statistiques

Ce notebook constitue la quatrième étape du projet de prédiction PL & qualification LDC.

**Objectifs** :
1. Valider formellement les observations de l'EDA avec des tests statistiques appropriés
2. Vérifier les hypothèses de normalité et homogénéité des variances
3. Choisir tests paramétriques vs non-paramétriques selon respect des hypothèses
4. Appliquer corrections pour comparaisons multiples (Bonferroni, FDR)
5. Quantifier les tailles d'effet (Cohen's d, Cliff's delta, etc.)
6. Établir la significativité statistique des patterns identifiés

**Approche adaptative** (PROJECT_CONTEXT section 5) :
- **Étape 1** : Vérifier hypothèses (normalité : Shapiro-Wilk/D'Agostino ; homogénéité : Levene)
- **Étape 2** : Si respectées → tests paramétriques (t-test, ANOVA)
- **Étape 3** : Si violées → tests non-paramétriques (Mann-Whitney, Kruskal-Wallis, chi²)
- **Étape 4** : Corrections multiples si nécessaire

**Tests à effectuer** :
1. Impact de la forme récente sur les résultats
2. Impact des séries (streaks) sur les résultats
3. Impact du classement sur les performances
4. Différence Top 4 vs Reste (points, buts, défense)
5. Avantage à domicile significatif ?
6. Association forme × classement (test d'indépendance)
7. Stabilité temporelle (début/milieu/fin saison)

---

## Note méthodologique

Les tests statistiques valideront ou invalideront les patterns observés dans les notebooks 02 et 03. Un pattern visuellement fort mais non significatif statistiquement (p > 0.05) devra être utilisé avec prudence dans la modélisation.

Les tailles d'effet permettront de distinguer **significativité statistique** (p-value) de **significativité pratique** (magnitude de l'effet).

## 4.1 — Imports et chargement

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Tests statistiques
from scipy import stats
from scipy.stats import shapiro, levene, mannwhitneyu, kruskal, chi2_contingency
from scipy.stats import ttest_ind, f_oneway
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

# Configuration graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Détection racine projet
import os
CURRENT_DIR = Path(os.getcwd())
PROJECT_ROOT = CURRENT_DIR
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# Chemins
CLEANED_DATA = PROJECT_ROOT / "data" / "interim" / "pl_cleaned.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
STATS_RESULTS_DIR = PROJECT_ROOT / "reports" / "stats_results"
STATS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Chargement depuis : {CLEANED_DATA}")
print(f"Résultats stats exportés vers : {STATS_RESULTS_DIR}")


Racine projet : /workspace
Chargement depuis : /workspace/data/interim/pl_cleaned.csv
Résultats stats exportés vers : /workspace/reports/stats_results


## 4.2 Chargement et préparation du datase

In [2]:
# Chargement
df = pd.read_csv(CLEANED_DATA)

# Conversion types
df['Date'] = pd.to_datetime(df['Date'])
df['Season'] = df['Season'].astype(str)
df['FTR'] = df['FTR'].astype('category')
df['HTR'] = df['HTR'].astype('category')

# Recréation Season_Year pour tri chronologique
def season_to_year(season_code):
    if pd.isna(season_code):
        return None
    season_str = str(season_code).zfill(4)
    year_start = int(season_str[:2])
    return 1900 + year_start if year_start >= 93 else 2000 + year_start

df['Season_Year'] = df['Season'].apply(season_to_year)

# Tri chronologique
df = df.sort_values(['Season_Year', 'Date']).reset_index(drop=True)

print(f"Dataset chargé : {df.shape}")
print(f"Période : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Saisons : {df['Season'].nunique()}")
print(f"\nAperçu colonnes :")
print(df.columns.tolist())


Dataset chargé : (12614, 27)
Période : 1993-08-14 → 2026-05-24
Saisons : 33

Aperçu colonnes :
['Date', 'Season', 'HomeTeam', 'AwayTeam', 'Referee', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'AvgH', 'AvgD', 'AvgA', 'Season_Year']


## 4.3 Expansion du dataset (une ligne par équipe par match)

In [4]:
# Fonction pour calculer les points d'un match
def get_points(ftr, is_home):
    """Retourne les points gagnés : 3 pour victoire, 1 pour nul, 0 pour défaite"""
    if pd.isna(ftr):
        return np.nan
    if is_home:
        return 3 if ftr == 'H' else (1 if ftr == 'D' else 0)
    else:
        return 3 if ftr == 'A' else (1 if ftr == 'D' else 0)

# Créer une ligne par équipe par match
matches_expanded = []

for _, row in df.iterrows():
    # Match domicile
    matches_expanded.append({
        'Date': row['Date'],
        'Season': row['Season'],
        'Season_Year': row['Season_Year'],
        'Team': row['HomeTeam'],
        'Opponent': row['AwayTeam'],
        'IsHome': True,
        'FTR': row['FTR'],
        'Goals_For': row['FTHG'],
        'Goals_Against': row['FTAG'],
        'Points': get_points(row['FTR'], True)
    })
    
    # Match extérieur
    matches_expanded.append({
        'Date': row['Date'],
        'Season': row['Season'],
        'Season_Year': row['Season_Year'],
        'Team': row['AwayTeam'],
        'Opponent': row['HomeTeam'],
        'IsHome': False,
        'FTR': row['FTR'],
        'Goals_For': row['FTAG'],
        'Goals_Against': row['FTHG'],
        'Points': get_points(row['FTR'], False)
    })

df_expanded = pd.DataFrame(matches_expanded)
df_expanded = df_expanded.sort_values(['Team', 'Season_Year', 'Date']).reset_index(drop=True)

print(f"Dataset expandé : {df_expanded.shape}")
print(f"Équipes uniques : {df_expanded['Team'].nunique()}")
print(f"\n✓ Chaque match est maintenant représenté par 2 lignes (une par équipe)")
print(f"  → {len(df)} matchs × 2 = {len(df_expanded)} lignes")


Dataset expandé : (25228, 10)
Équipes uniques : 51

✓ Chaque match est maintenant représenté par 2 lignes (une par équipe)
  → 12614 matchs × 2 = 25228 lignes


In [5]:
# Calcul de la forme récente (comme dans notebook 03)
print("Calcul de la forme récente (rolling 5 matchs)...")

df_expanded['Form_Last5'] = np.nan

for team in df_expanded['Team'].unique():
    for season in df_expanded['Season'].unique():
        mask = (df_expanded['Team'] == team) & (df_expanded['Season'] == season)
        team_season_data = df_expanded[mask].copy()
        
        if len(team_season_data) > 0:
            # Rolling sum sur 5 matchs
            rolling_points = team_season_data['Points'].rolling(window=5, min_periods=1).sum()
            # Shift de 1 pour avoir la forme AVANT le match
            rolling_points_shifted = rolling_points.shift(1)
            df_expanded.loc[mask, 'Form_Last5'] = rolling_points_shifted.values

# Catégorisation de la forme
df_expanded['Form_Category'] = pd.cut(
    df_expanded['Form_Last5'], 
    bins=[-0.1, 3, 6, 9, 15], 
    labels=['Mauvaise (0-3)', 'Moyenne (4-6)', 'Bonne (7-9)', 'Excellente (10-15)']
)

# Suppression première ligne par équipe/saison (Form_Last5 = NaN)
df_expanded = df_expanded.dropna(subset=['Form_Last5']).copy()

print("✓ Forme calculée et catégorisée")
print(f"\nMatchs avec forme disponible : {len(df_expanded)}")
print(f"\nDistribution des catégories de forme :")
print(df_expanded['Form_Category'].value_counts().sort_index())


Calcul de la forme récente (rolling 5 matchs)...
✓ Forme calculée et catégorisée

Matchs avec forme disponible : 24564

Distribution des catégories de forme :
Form_Category
Mauvaise (0-3)        5007
Moyenne (4-6)         7602
Bonne (7-9)           6812
Excellente (10-15)    5143
Name: count, dtype: int64


In [6]:
print("=" * 70)
print("TEST 1 : IMPACT DE LA FORME RÉCENTE SUR LES RÉSULTATS")
print("=" * 70)

# Hypothèse : La forme récente influence significativement la probabilité de victoire

# 1. Préparation des données
# Variable dépendante : Victoire (Points = 3)
df_expanded['Victory'] = (df_expanded['Points'] == 3).astype(int)

# Grouper par catégorie de forme
form_groups = []
for category in ['Mauvaise (0-3)', 'Moyenne (4-6)', 'Bonne (7-9)', 'Excellente (10-15)']:
    victories = df_expanded[df_expanded['Form_Category'] == category]['Victory']
    form_groups.append(victories)

print("\n1. STATISTIQUES DESCRIPTIVES PAR FORME")
print("-" * 70)
for i, category in enumerate(['Mauvaise (0-3)', 'Moyenne (4-6)', 'Bonne (7-9)', 'Excellente (10-15)']):
    data = form_groups[i]
    win_rate = data.mean() * 100
    n = len(data)
    print(f"{category:20} | n={n:5} | Taux victoire={win_rate:5.2f}% | Moyenne points={data.sum()/len(data)*3:.2f}")

print("\n2. VÉRIFICATION HYPOTHÈSES (normalité)")
print("-" * 70)
# Test de Shapiro-Wilk pour chaque groupe (si n < 5000, sinon D'Agostino)
for i, category in enumerate(['Mauvaise (0-3)', 'Moyenne (4-6)', 'Bonne (7-9)', 'Excellente (10-15)']):
    data = form_groups[i]
    if len(data) < 5000:
        stat, p_value = shapiro(data.sample(min(5000, len(data)), random_state=42))
        test_name = "Shapiro-Wilk"
    else:
        from scipy.stats import normaltest
        stat, p_value = normaltest(data)
        test_name = "D'Agostino"
    
    normal = "OUI" if p_value > 0.05 else "NON"
    print(f"{category:20} | {test_name:15} | p-value={p_value:.4f} | Normal={normal}")

print("\n3. TEST D'HOMOGÉNÉITÉ DES VARIANCES (Levene)")
print("-" * 70)
stat_levene, p_levene = levene(*form_groups)
homogeneous = "OUI" if p_levene > 0.05 else "NON"
print(f"Statistique de Levene = {stat_levene:.4f}")
print(f"p-value = {p_levene:.4f}")
print(f"Variances homogènes ? {homogeneous}")

print("\n" + "=" * 70)
print("DÉCISION SUR LE TEST À UTILISER :")
print("=" * 70)

TEST 1 : IMPACT DE LA FORME RÉCENTE SUR LES RÉSULTATS

1. STATISTIQUES DESCRIPTIVES PAR FORME
----------------------------------------------------------------------
Mauvaise (0-3)       | n= 5007 | Taux victoire=30.42% | Moyenne points=0.91
Moyenne (4-6)        | n= 7602 | Taux victoire=33.52% | Moyenne points=1.01
Bonne (7-9)          | n= 6812 | Taux victoire=38.43% | Moyenne points=1.15
Excellente (10-15)   | n= 5143 | Taux victoire=47.81% | Moyenne points=1.43

2. VÉRIFICATION HYPOTHÈSES (normalité)
----------------------------------------------------------------------
Mauvaise (0-3)       | D'Agostino      | p-value=0.0000 | Normal=NON
Moyenne (4-6)        | D'Agostino      | p-value=0.0000 | Normal=NON
Bonne (7-9)          | D'Agostino      | p-value=0.0000 | Normal=NON
Excellente (10-15)   | D'Agostino      | p-value=0.0000 | Normal=NON

3. TEST D'HOMOGÉNÉITÉ DES VARIANCES (Levene)
----------------------------------------------------------------------
Statistique de Levene = 133

## 4.5 Test de Kruskal-Wallis + taille d'effet

In [7]:
print("HYPOTHÈSES NON RESPECTÉES → Test non-paramétrique requis")
print("\n4. TEST DE KRUSKAL-WALLIS (comparaison 4 groupes)")
print("-" * 70)

# Test de Kruskal-Wallis (équivalent non-paramétrique de l'ANOVA)
h_stat, p_kruskal = kruskal(*form_groups)

print(f"Statistique H de Kruskal-Wallis = {h_stat:.4f}")
print(f"p-value = {p_kruskal:.10f}")
print(f"\nRésultat : {'✓ SIGNIFICATIF' if p_kruskal < 0.05 else '✗ NON SIGNIFICATIF'} (seuil α = 0.05)")

if p_kruskal < 0.05:
    print("\n→ Il existe au moins une différence significative entre les groupes de forme.")
    print("→ La forme récente influence significativement les résultats.")

print("\n5. TESTS POST-HOC (comparaisons par paires avec correction Bonferroni)")
print("-" * 70)

# Comparaisons par paires avec Mann-Whitney U
categories = ['Mauvaise (0-3)', 'Moyenne (4-6)', 'Bonne (7-9)', 'Excellente (10-15)']
comparisons = []

for i in range(len(categories)):
    for j in range(i+1, len(categories)):
        u_stat, p_value = mannwhitneyu(form_groups[i], form_groups[j], alternative='two-sided')
        comparisons.append({
            'Groupe 1': categories[i],
            'Groupe 2': categories[j],
            'U': u_stat,
            'p-value': p_value,
            'p-ajusté (Bonferroni)': p_value * 6  # 6 comparaisons (4 choose 2)
        })

print(f"{'Comparaison':40} | {'U-stat':12} | {'p-value':10} | p-ajusté | Significatif")
print("-" * 90)
for comp in comparisons:
    p_adj = comp['p-ajusté (Bonferroni)']
    p_adj_display = min(p_adj, 1.0)  # Cap à 1.0
    signif = "✓ OUI" if p_adj < 0.05 else "✗ NON"
    print(f"{comp['Groupe 1']:20} vs {comp['Groupe 2']:20} | {comp['U']:12.0f} | {comp['p-value']:.4e} | {p_adj_display:.4f}   | {signif}")

print("\n6. TAILLE D'EFFET (Eta² pour Kruskal-Wallis)")
print("-" * 70)
# Calcul de Eta² (approximation via H-statistique)
n_total = sum(len(g) for g in form_groups)
eta_squared = (h_stat - len(form_groups) + 1) / (n_total - len(form_groups))

print(f"Eta² = {eta_squared:.4f}")
if eta_squared < 0.01:
    effet = "négligeable"
elif eta_squared < 0.06:
    effet = "petit"
elif eta_squared < 0.14:
    effet = "moyen"
else:
    effet = "large"
print(f"Interprétation : Taille d'effet {effet}")

print("\n" + "=" * 70)
print("CONCLUSION TEST 1")
print("=" * 70)


HYPOTHÈSES NON RESPECTÉES → Test non-paramétrique requis

4. TEST DE KRUSKAL-WALLIS (comparaison 4 groupes)
----------------------------------------------------------------------
Statistique H de Kruskal-Wallis = 394.8832
p-value = 0.0000000000

Résultat : ✓ SIGNIFICATIF (seuil α = 0.05)

→ Il existe au moins une différence significative entre les groupes de forme.
→ La forme récente influence significativement les résultats.

5. TESTS POST-HOC (comparaisons par paires avec correction Bonferroni)
----------------------------------------------------------------------
Comparaison                              | U-stat       | p-value    | p-ajusté | Significatif
------------------------------------------------------------------------------------------
Mauvaise (0-3)       vs Moyenne (4-6)        |     18441612 | 2.6983e-04 | 0.0016   | ✓ OUI
Mauvaise (0-3)       vs Bonne (7-9)          |     15687017 | 1.8090e-19 | 0.0000   | ✓ OUI
Mauvaise (0-3)       vs Excellente (10-15)   |     106357

In [8]:
print("✓ La forme récente est un facteur prédictif VALIDÉ statistiquement.")
print("✓ Toutes les différences entre catégories sont significatives.")
print("✓ Variable à inclure prioritairement dans le modèle prédictif.")

# Export des résultats dans un fichier texte
results_file = STATS_RESULTS_DIR / "test1_forme_recente.txt"
with open(results_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("TEST 1 : IMPACT DE LA FORME RÉCENTE SUR LES RÉSULTATS\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Test de Kruskal-Wallis : H = {h_stat:.4f}, p < 0.001\n")
    f.write(f"Taille d'effet (Eta²) : {eta_squared:.4f} (petit)\n\n")
    f.write("Tests post-hoc (Mann-Whitney U avec correction Bonferroni) :\n")
    for comp in comparisons:
        signif = "SIGNIFICATIF" if comp['p-ajusté (Bonferroni)'] < 0.05 else "NON SIGNIFICATIF"
        f.write(f"  {comp['Groupe 1']} vs {comp['Groupe 2']} : {signif}\n")
    f.write("\nCONCLUSION : La forme récente influence significativement les résultats.\n")
    f.write("Variable validée pour inclusion dans le modèle prédictif.\n")

print(f"\n✓ Résultats exportés vers : {results_file}")
print("\n" + "=" * 70)
print("Prêt pour le TEST 2 : Impact des séries (Streaks)")
print("=" * 70)


✓ La forme récente est un facteur prédictif VALIDÉ statistiquement.
✓ Toutes les différences entre catégories sont significatives.
✓ Variable à inclure prioritairement dans le modèle prédictif.

✓ Résultats exportés vers : /workspace/reports/stats_results/test1_forme_recente.txt

Prêt pour le TEST 2 : Impact des séries (Streaks)


## 4.7 TEST 2 — Impact des séries (Streaks) sur les résultats

In [9]:
print("\n" + "=" * 70)
print("TEST 2 : IMPACT DES SÉRIES (STREAKS) SUR LES RÉSULTATS")
print("=" * 70)

# Calcul des séries de victoires/défaites consécutives
print("\n1. CALCUL DES SÉRIES (STREAKS)")
print("-" * 70)

df_expanded['Streak'] = 0

for team in df_expanded['Team'].unique():
    for season in df_expanded['Season'].unique():
        mask = (df_expanded['Team'] == team) & (df_expanded['Season'] == season)
        team_season_data = df_expanded[mask].copy()
        
        if len(team_season_data) > 0:
            streak = []
            current_streak = 0
            
            for _, row in team_season_data.iterrows():
                if row['Points'] == 3:  # Victoire
                    current_streak = current_streak + 1 if current_streak > 0 else 1
                elif row['Points'] == 0:  # Défaite
                    current_streak = current_streak - 1 if current_streak < 0 else -1
                else:  # Nul
                    current_streak = 0
                
                streak.append(current_streak)
            
            # Shift de 1 pour avoir la série AVANT le match
            streak_shifted = [0] + streak[:-1]
            df_expanded.loc[mask, 'Streak'] = streak_shifted

# Catégorisation des séries
df_expanded['Streak_Category'] = pd.cut(
    df_expanded['Streak'],
    bins=[-15, -3, 0, 3, 20],
    labels=['Série défaites (-3+)', 'Neutre/Mixte', 'Série victoires (1-3)', 'Série victoires (3+)']
)

print("✓ Séries calculées et catégorisées")
print(f"\nDistribution des catégories de séries :")
print(df_expanded['Streak_Category'].value_counts().sort_index())

print("\n2. STATISTIQUES DESCRIPTIVES PAR SÉRIE")
print("-" * 70)

streak_groups = []
for category in ['Série défaites (-3+)', 'Neutre/Mixte', 'Série victoires (1-3)', 'Série victoires (3+)']:
    victories = df_expanded[df_expanded['Streak_Category'] == category]['Victory']
    streak_groups.append(victories)
    win_rate = victories.mean() * 100
    n = len(victories)
    avg_points = victories.sum() / len(victories) * 3
    print(f"{category:30} | n={n:5} | Taux victoire={win_rate:5.2f}% | Moyenne points={avg_points:.2f}")

print("\n3. VÉRIFICATION HYPOTHÈSES")
print("-" * 70)

# Test normalité
for i, category in enumerate(['Série défaites (-3+)', 'Neutre/Mixte', 'Série victoires (1-3)', 'Série victoires (3+)']):
    data = streak_groups[i]
    if len(data) > 50:  # Seuil minimum
        from scipy.stats import normaltest
        stat, p_value = normaltest(data)
        normal = "OUI" if p_value > 0.05 else "NON"
        print(f"{category:30} | p-value={p_value:.4f} | Normal={normal}")

# Test homogénéité
stat_levene, p_levene = levene(*streak_groups)
homogeneous = "OUI" if p_levene > 0.05 else "NON"
print(f"\nLevene : p-value={p_levene:.4f} | Variances homogènes={homogeneous}")

print("\n4. TEST DE KRUSKAL-WALLIS")
print("-" * 70)

h_stat_streak, p_kruskal_streak = kruskal(*streak_groups)
print(f"Statistique H = {h_stat_streak:.4f}")
print(f"p-value = {p_kruskal_streak:.10f}")
print(f"\nRésultat : {'✓ SIGNIFICATIF' if p_kruskal_streak < 0.05 else '✗ NON SIGNIFICATIF'}")

# Taille d'effet
n_total_streak = sum(len(g) for g in streak_groups)
eta_squared_streak = (h_stat_streak - len(streak_groups) + 1) / (n_total_streak - len(streak_groups))
print(f"\nEta² = {eta_squared_streak:.4f}")

print("\n" + "=" * 70)



TEST 2 : IMPACT DES SÉRIES (STREAKS) SUR LES RÉSULTATS

1. CALCUL DES SÉRIES (STREAKS)
----------------------------------------------------------------------
✓ Séries calculées et catégorisées

Distribution des catégories de séries :
Streak_Category
Série défaites (-3+)      1452
Neutre/Mixte             14217
Série victoires (1-3)     8104
Série victoires (3+)       791
Name: count, dtype: int64

2. STATISTIQUES DESCRIPTIVES PAR SÉRIE
----------------------------------------------------------------------
Série défaites (-3+)           | n= 1452 | Taux victoire=28.86% | Moyenne points=0.87
Neutre/Mixte                   | n=14217 | Taux victoire=35.71% | Moyenne points=1.07
Série victoires (1-3)          | n= 8104 | Taux victoire=39.50% | Moyenne points=1.18
Série victoires (3+)           | n=  791 | Taux victoire=57.02% | Moyenne points=1.71

3. VÉRIFICATION HYPOTHÈSES
----------------------------------------------------------------------
Série défaites (-3+)           | p-value=0.00

##  Export TEST 2 + Préparation TEST 3 (Classement)

In [10]:
# Export résultats TEST 2
results_file = STATS_RESULTS_DIR / "test2_series_streaks.txt"
with open(results_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("TEST 2 : IMPACT DES SÉRIES (STREAKS) SUR LES RÉSULTATS\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Test de Kruskal-Wallis : H = {h_stat_streak:.4f}, p < 0.001\n")
    f.write(f"Taille d'effet (Eta²) : {eta_squared_streak:.4f} (négligeable)\n\n")
    f.write("NOTE MÉTHODOLOGIQUE :\n")
    f.write("L'Eta² faible reflète une distribution déséquilibrée (58% neutres).\n")
    f.write("Aux extrêmes, l'effet prédictif est massif (+97% d'écart).\n")
    f.write("→ Variable validée, usage conditionnel recommandé en modélisation.\n\n")
    f.write("CONCLUSION : Les séries influencent significativement les résultats,\n")
    f.write("avec un impact très fort en situations extrêmes (±3 matchs).\n")

print(f"✓ Résultats TEST 2 exportés vers : {results_file}")

print("\n" + "=" * 70)
print("TEST 3 : IMPACT DU CLASSEMENT SUR LES PERFORMANCES")
print("=" * 70)

# Calcul du classement (cumul points par équipe/saison)
print("\n1. CALCUL DU CLASSEMENT")
print("-" * 70)

df_expanded['Cumul_Points'] = 0
df_expanded['Rank'] = 0

for season in df_expanded['Season'].unique():
    season_mask = df_expanded['Season'] == season
    season_data = df_expanded[season_mask].copy()
    
    for team in season_data['Team'].unique():
        team_season_mask = (df_expanded['Season'] == season) & (df_expanded['Team'] == team)
        team_season_data = df_expanded[team_season_mask].copy()
        
        # Cumul progressif des points
        cumul_points = team_season_data['Points'].cumsum()
        # Shift de 1 pour avoir le classement AVANT le match
        cumul_points_shifted = cumul_points.shift(1).fillna(0)
        df_expanded.loc[team_season_mask, 'Cumul_Points'] = cumul_points_shifted
    
    # Calcul du rang après chaque journée
    for idx in df_expanded[season_mask].index:
        date = df_expanded.loc[idx, 'Date']
        # Rang basé sur les points cumulés à cette date
        teams_at_date = df_expanded[(df_expanded['Season'] == season) & (df_expanded['Date'] <= date)]
        latest_points = teams_at_date.groupby('Team')['Cumul_Points'].last()
        rank = (latest_points.rank(ascending=False, method='min'))
        team = df_expanded.loc[idx, 'Team']
        if team in rank.index:
            df_expanded.loc[idx, 'Rank'] = rank[team]

# Catégorisation du classement
df_expanded['Rank_Category'] = pd.cut(
    df_expanded['Rank'],
    bins=[0, 4, 14, 25],
    labels=['Top 4', 'Milieu (5-14)', 'Bas (15+)']
)

# Suppression des premières journées (Rank = 0 ou non défini)
df_rank = df_expanded[df_expanded['Rank'] > 0].copy()

print(f"✓ Classement calculé pour {len(df_rank)} matchs")
print(f"\nDistribution des catégories de classement :")
print(df_rank['Rank_Category'].value_counts().sort_index())


✓ Résultats TEST 2 exportés vers : /workspace/reports/stats_results/test2_series_streaks.txt

TEST 3 : IMPACT DU CLASSEMENT SUR LES PERFORMANCES

1. CALCUL DU CLASSEMENT
----------------------------------------------------------------------
✓ Classement calculé pour 24564 matchs

Distribution des catégories de classement :
Rank_Category
Top 4             6279
Milieu (5-14)    12125
Bas (15+)         6160
Name: count, dtype: int64


## TEST 3 complet — Impact du classement

In [12]:
print("\n2. STATISTIQUES DESCRIPTIVES PAR CLASSEMENT")
print("-" * 70)

rank_groups = []
for category in ['Top 4', 'Milieu (5-14)', 'Bas (15+)']:
    victories = df_rank[df_rank['Rank_Category'] == category]['Victory']
    rank_groups.append(victories)
    win_rate = victories.mean() * 100
    n = len(victories)
    avg_points = victories.sum() / len(victories) * 3
    print(f"{category:20} | n={n:5} | Taux victoire={win_rate:5.2f}% | Moyenne points={avg_points:.2f}")

print("\n3. VÉRIFICATION HYPOTHÈSES")
print("-" * 70)

# Test normalité
for i, category in enumerate(['Top 4', 'Milieu (5-14)', 'Bas (15+)']):
    data = rank_groups[i]
    from scipy.stats import normaltest
    stat, p_value = normaltest(data)
    normal = "OUI" if p_value > 0.05 else "NON"
    print(f"{category:20} | p-value={p_value:.4f} | Normal={normal}")

# Test homogénéité
stat_levene_rank, p_levene_rank = levene(*rank_groups)
homogeneous = "OUI" if p_levene_rank > 0.05 else "NON"
print(f"\nLevene : p-value={p_levene_rank:.4f} | Variances homogènes={homogeneous}")

print("\n4. TEST DE KRUSKAL-WALLIS")
print("-" * 70)

h_stat_rank, p_kruskal_rank = kruskal(*rank_groups)
print(f"Statistique H = {h_stat_rank:.4f}")
print(f"p-value = {p_kruskal_rank:.10f}")
print(f"\nRésultat : {'✓ SIGNIFICATIF' if p_kruskal_rank < 0.05 else '✗ NON SIGNIFICATIF'}")

# Taille d'effet
n_total_rank = sum(len(g) for g in rank_groups)
eta_squared_rank = (h_stat_rank - len(rank_groups) + 1) / (n_total_rank - len(rank_groups))
print(f"\nEta² = {eta_squared_rank:.4f}")

if eta_squared_rank < 0.01:
    effet = "négligeable"
elif eta_squared_rank < 0.06:
    effet = "petit"
else:
    effet = "moyen/large"
print(f"Interprétation : Taille d'effet {effet}")

print("\n5. TESTS POST-HOC (Mann-Whitney U avec correction Bonferroni)")
print("-" * 70)

categories_rank = ['Top 4', 'Milieu (5-14)', 'Bas (15+)']
comparisons_rank = []

for i in range(len(categories_rank)):
    for j in range(i+1, len(categories_rank)):
        u_stat, p_value = mannwhitneyu(rank_groups[i], rank_groups[j], alternative='two-sided')
        comparisons_rank.append({
            'Groupe 1': categories_rank[i],
            'Groupe 2': categories_rank[j],
            'p-value': p_value,
            'p-ajusté': p_value * 3  # 3 comparaisons
        })

print(f"{'Comparaison':40} | {'p-value':10} | p-ajusté | Significatif")
print("-" * 80)
for comp in comparisons_rank:
    p_adj = min(comp['p-ajusté'], 1.0)
    signif = "✓ OUI" if p_adj < 0.05 else "✗ NON"
    print(f"{comp['Groupe 1']:20} vs {comp['Groupe 2']:20} | {comp['p-value']:.4e} | {p_adj:.4f}   | {signif}")

# Export résultats
results_file = STATS_RESULTS_DIR / "test3_classement.txt"
with open(results_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("TEST 3 : IMPACT DU CLASSEMENT SUR LES PERFORMANCES\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Test de Kruskal-Wallis : H = {h_stat_rank:.4f}, p < 0.001\n")
    f.write(f"Taille d'effet (Eta²) : {eta_squared_rank:.4f} ({effet})\n\n")
    f.write("Tests post-hoc (Mann-Whitney U avec Bonferroni) :\n")
    for comp in comparisons_rank:
        signif = "SIGNIFICATIF" if comp['p-ajusté'] < 0.05 else "NON SIGNIFICATIF"
        f.write(f"  {comp['Groupe 1']} vs {comp['Groupe 2']} : {signif}\n")
    f.write("\nCONCLUSION : Le classement influence significativement les résultats,\n")
    f.write("mais avec un effet modéré (+20% Top 4 vs Bas selon EDA).\n")

print(f"\n✓ Résultats TEST 3 exportés vers : {results_file}")
print("\n" + "=" * 70)
print("Tests 1-2-3 terminés. Prochaine étape : Tests spécifiques (Top 4, domicile)")
print("=" * 70)



2. STATISTIQUES DESCRIPTIVES PAR CLASSEMENT
----------------------------------------------------------------------
Top 4                | n= 6279 | Taux victoire=50.66% | Moyenne points=1.52
Milieu (5-14)        | n=12125 | Taux victoire=35.04% | Moyenne points=1.05
Bas (15+)            | n= 6160 | Taux victoire=27.91% | Moyenne points=0.84

3. VÉRIFICATION HYPOTHÈSES
----------------------------------------------------------------------
Top 4                | p-value=0.0000 | Normal=NON
Milieu (5-14)        | p-value=0.0000 | Normal=NON
Bas (15+)            | p-value=0.0000 | Normal=NON

Levene : p-value=0.0000 | Variances homogènes=NON

4. TEST DE KRUSKAL-WALLIS
----------------------------------------------------------------------
Statistique H = 738.7239
p-value = 0.0000000000

Résultat : ✓ SIGNIFICATIF

Eta² = 0.0300
Interprétation : Taille d'effet petit

5. TESTS POST-HOC (Mann-Whitney U avec correction Bonferroni)
----------------------------------------------------------------

## TEST 4 — Avantage à domicile (test simple)

In [13]:
print("\n" + "=" * 70)
print("TEST 4 : AVANTAGE À DOMICILE")
print("=" * 70)

print("\n1. STATISTIQUES DESCRIPTIVES")
print("-" * 70)

home_victories = df_expanded[df_expanded['IsHome'] == True]['Victory']
away_victories = df_expanded[df_expanded['IsHome'] == False]['Victory']

home_win_rate = home_victories.mean() * 100
away_win_rate = away_victories.mean() * 100

print(f"Domicile : n={len(home_victories):5} | Taux victoire={home_win_rate:.2f}%")
print(f"Extérieur : n={len(away_victories):5} | Taux victoire={away_win_rate:.2f}%")
print(f"\nÉcart : +{home_win_rate - away_win_rate:.2f} points de % en faveur du domicile")

print("\n2. TEST DE MANN-WHITNEY U (comparaison 2 groupes indépendants)")
print("-" * 70)

u_stat_home, p_home = mannwhitneyu(home_victories, away_victories, alternative='two-sided')

print(f"Statistique U = {u_stat_home:.0f}")
print(f"p-value = {p_home:.10f}")
print(f"\nRésultat : {'✓ SIGNIFICATIF' if p_home < 0.05 else '✗ NON SIGNIFICATIF'}")

# Taille d'effet (Cohen's d pour données binaires)
from numpy import sqrt
mean_diff = home_victories.mean() - away_victories.mean()
pooled_std = sqrt((home_victories.std()**2 + away_victories.std()**2) / 2)
cohens_d = mean_diff / pooled_std

print(f"\n3. TAILLE D'EFFET (Cohen's d)")
print("-" * 70)
print(f"Cohen's d = {cohens_d:.4f}")

if abs(cohens_d) < 0.2:
    effet = "négligeable"
elif abs(cohens_d) < 0.5:
    effet = "petit"
elif abs(cohens_d) < 0.8:
    effet = "moyen"
else:
    effet = "large"
print(f"Interprétation : Taille d'effet {effet}")

# Export
results_file = STATS_RESULTS_DIR / "test4_avantage_domicile.txt"
with open(results_file, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("TEST 4 : AVANTAGE À DOMICILE\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Test de Mann-Whitney U : U = {u_stat_home:.0f}, p < 0.001\n")
    f.write(f"Taille d'effet (Cohen's d) : {cohens_d:.4f} ({effet})\n\n")
    f.write(f"Écart observé : +{home_win_rate - away_win_rate:.2f}% de victoires à domicile\n\n")
    f.write("CONCLUSION : L'avantage à domicile est significatif et robuste.\n")

print(f"\n✓ Résultats TEST 4 exportés vers : {results_file}")
print("\n" + "=" * 70)
print("4 tests terminés. Résumé final à venir.")
print("=" * 70)



TEST 4 : AVANTAGE À DOMICILE

1. STATISTIQUES DESCRIPTIVES
----------------------------------------------------------------------
Domicile : n=12282 | Taux victoire=45.76%
Extérieur : n=12282 | Taux victoire=28.72%

Écart : +17.03 points de % en faveur du domicile

2. TEST DE MANN-WHITNEY U (comparaison 2 groupes indépendants)
----------------------------------------------------------------------
Statistique U = 88270734
p-value = 0.0000000000

Résultat : ✓ SIGNIFICATIF

3. TAILLE D'EFFET (Cohen's d)
----------------------------------------------------------------------
Cohen's d = 0.3579
Interprétation : Taille d'effet petit

✓ Résultats TEST 4 exportés vers : /workspace/reports/stats_results/test4_avantage_domicile.txt

4 tests terminés. Résumé final à venir.
